# Phase 4 Validation: Q&A Engine

Tests core Q&A scenarios including single-turn, multi-turn, and not-found handling.

**Prerequisites**: Anthropic API key set in `.env`.

In [ ]:
import contextlib
import time
from dataclasses import dataclass
from pathlib import Path

from credit_analyzer.generation.qa_engine import QAEngine, QAResponse
from credit_analyzer.llm.factory import get_provider
from credit_analyzer.processing.chunker import Chunker, build_search_text
from credit_analyzer.processing.definitions import DefinitionsParser
from credit_analyzer.processing.pdf_extractor import PDFExtractor
from credit_analyzer.processing.section_detector import SectionDetector
from credit_analyzer.retrieval.bm25_store import BM25Store
from credit_analyzer.retrieval.embedder import Embedder
from credit_analyzer.retrieval.hybrid_retriever import HybridRetriever
from credit_analyzer.retrieval.vector_store import VectorStore

PDF_PATH = Path(r"../credit_agreement.pdf")
CHROMA_DIR = str(Path(r"../chroma_validation_p4"))
DOC_ID = "ribbon_p4"

## Pipeline Setup (Phases 1-3)

In [ ]:
# Phase 1: Extract, detect, parse, chunk
t0 = time.time()

extractor = PDFExtractor()
doc = extractor.extract(PDF_PATH)

detector = SectionDetector()
sections = detector.detect_sections(doc)

defn_sections = [s for s in sections if s.section_type == "definitions"]
if not defn_sections:
    msg = "No definitions section found"
    raise RuntimeError(msg)
parser = DefinitionsParser()
defn_index = parser.parse(defn_sections[0])

chunker = Chunker()
chunks = chunker.chunk_document(sections, defn_index)

preamble_sections = [s for s in sections if s.section_type == "preamble"]
preamble_text = preamble_sections[0].text if preamble_sections else None

print(f"Phase 1: {time.time() - t0:.1f}s")
print(f"  Pages: {doc.total_pages} | Sections: {len(sections)} | Terms: {len(defn_index.definitions)} | Chunks: {len(chunks)}")

In [ ]:
# Phase 2: LLM provider
llm = get_provider()
print(f"Provider: {llm.model_name()} | Available: {llm.is_available()}")

In [ ]:
# Phase 3: Embed, index, build retriever
t0 = time.time()
embedder = Embedder()
embeddings = embedder.embed([build_search_text(c) for c in chunks])
print(f"Embedding: {time.time() - t0:.1f}s")

store = VectorStore(persist_directory=CHROMA_DIR)
with contextlib.suppress(Exception):
    store.delete_collection(DOC_ID)
store.create_collection(DOC_ID)
store.add_chunks(DOC_ID, chunks, embeddings)

bm25 = BM25Store()
bm25.build_index(chunks)

retriever = HybridRetriever(
    vector_store=store,
    bm25_store=bm25,
    embedder=embedder,
    definitions_index=defn_index,
)

qa = QAEngine(retriever=retriever, llm=llm)
if preamble_text:
    qa.set_preamble(preamble_text)

print("Pipeline ready")

## Test Harness

In [ ]:
@dataclass
class TestResult:
    """Result of a single Q&A validation test."""

    name: str
    question: str
    response: QAResponse
    elapsed: float


ALL_RESULTS: list[TestResult] = []


def run_test(
    name: str,
    question: str,
    *,
    clear_history: bool = True,
) -> QAResponse:
    """Run a single Q&A test and print results."""
    if clear_history:
        qa.clear_history()

    print(f"=== {name} ===")
    print(f"Q: {question}\n")

    t0 = time.time()
    resp = qa.ask(question, DOC_ID)
    elapsed = time.time() - t0

    ALL_RESULTS.append(TestResult(name=name, question=question, response=resp, elapsed=elapsed))

    print(f"{resp.answer}\n")
    print(f"Confidence: {resp.confidence}")
    print(f"Sources ({len(resp.sources)}):")
    for s in resp.sources:
        pages = ", ".join(str(p) for p in s.page_numbers)
        print(f"  - Section {s.section_id}: {s.section_title} (pp. {pages})")
    print(f"Chunks: {len(resp.retrieved_chunks)} | LLM: {resp.llm_response.duration_seconds:.1f}s | Tokens: {resp.llm_response.tokens_used} | Total: {elapsed:.1f}s")
    print()
    return resp

## Core Q&A Tests

In [ ]:
# T1: Revolving commitment amount (exact dollar figure from preamble + definitions)
run_test(
    "Revolving Commitment",
    "What is the total revolving commitment amount?",
)

In [ ]:
# T2: Financial covenants (multi-chunk section with ratio table)
run_test(
    "Financial Covenants",
    "What financial covenants are included in the agreement and what are the specific ratio thresholds?",
)

In [ ]:
# T3: Incremental debt capacity (core use case - needs Section 2.27 + definition chunks)
run_test(
    "Incremental Debt",
    "How much incremental debt can the borrower incur? Include the fixed dollar basket, ratio-based capacity, and any key conditions.",
)

In [ ]:
# T4: Restricted payments (broad section with many carve-outs)
run_test(
    "Restricted Payments",
    "What restricted payments are permitted under the credit agreement?",
)

In [ ]:
# T5: Interest rate (needs Applicable Margin pricing grid - promoted definition)
run_test(
    "Interest Rate",
    "What is the interest rate on the term loan?",
)

In [ ]:
# T6: Multi-turn follow-up (uses history from T5 - do NOT clear history)
run_test(
    "Interest Rate Follow-Up",
    "What about the revolving facility? Is the pricing the same?",
    clear_history=False,
)

In [ ]:
# T7: Not-found test (question about something not in a typical credit agreement)
run_test(
    "Not Found",
    "What borrowing base formula determines revolver availability?"
)

## Validation Summary

In [ ]:
print("=" * 70)
print("PHASE 4 VALIDATION SUMMARY")
print("=" * 70)
print(f"Total tests: {len(ALL_RESULTS)}")
print(f"Total time:  {sum(r.elapsed for r in ALL_RESULTS):.1f}s")
print(f"Avg LLM time: {sum(r.response.llm_response.duration_seconds for r in ALL_RESULTS) / len(ALL_RESULTS):.1f}s")
print()

flags: list[str] = []

print(f"{'Test':<25} {'Confidence':<12} {'Chunks':<8} {'Sources':<9} {'LLM (s)':<9} {'Tokens':<8}")
print("-" * 70)
for r in ALL_RESULTS:
    resp = r.response
    conf = resp.confidence
    conf_display = conf
    if conf in ("MEDIUM", "LOW"):
        conf_display = f"{conf} ***"
        flags.append(f"{r.name}: {conf} confidence")
    print(
        f"{r.name:<25} {conf_display:<12} {len(resp.retrieved_chunks):<8} "
        f"{len(resp.sources):<9} {resp.llm_response.duration_seconds:<9.1f} {resp.llm_response.tokens_used:<8}"
    )

print()
if flags:
    print("FLAGS:")
    for f in flags:
        print(f"  - {f}")
else:
    print("No flags. All tests returned HIGH confidence.")

print()
print("Review each answer above for correctness before marking Phase 4 complete.")